# 06 — Entrenamiento exclusivo del mask decoder

Solo Colab GPU, ejecutado por el usuario. Leer docs/project/validacion_etapa6.md. Etapa abierta hasta revisión de evidencia; sin reservados, ablaciones adicionales ni demo.


## Protocolo previo
10 épocas por régimen, semilla 42, AdamW LR 1e-5 / WD 0.01, FP32, batch efectivo 4 por acumulación. SAM base independiente, encoders congelados, DINO no cargado: features/probe propios de etapa 5. Prompts fijos. BCE+Dice válida de unión suave por máximo; sin componentes = vacío sin gradiente. Best por IoU agua val, primer empate. Se conservan resultados negativos.


## 1. Paquete y entorno
Exportar scripts/export_stage6_bundle.py en la PC; subir ZIP a MyDrive y copiar SHA. GPU requerida. Raíz histórica íntegra de Drive. Nunca instalar ni ejecutar modelos en la PC.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage6')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage6.zip')
EXPECTED_BUNDLE_SHA256 = '3ebf75e5287a765e8fb2673ec55b74fa89d2934cca2b8fc2ce935758468df2c5'  # Copiar salida de export_stage6_bundle.py.
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Token opcional para descarga autorizada, nunca imprimirlo ni incluirlo en informes.
from google.colab import userdata
try:
    child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print('HF_TOKEN no configurado; se usara la cache autorizada si existe.')
LOG_DIR = Path(PERSIST_ROOT) / 'stage6/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    import time
    log_path = LOG_DIR / (str(time.time_ns()) + '_' + log_name)
    print('Log persistente:', log_path)
    with log_path.open('x', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_path}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


Mounted at /content/drive
Log persistente: /content/drive/MyDrive/auto-prompt-sam2/stage6/notebook_logs/1793564756888280059_bootstrap.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 27.5 MB/s  0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 26.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 109.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 165.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Interrupción controlada
Ejecutar prepare UNA VEZ. Una época por régimen, con last persistente. Guardar notebook con salidas; después desconectar y eliminar runtime. No ejecutar resume en este runtime.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_stage6.py'), 'prepare', '--root', PERSIST_ROOT], 'prepare.log')


Log persistente: /content/drive/MyDrive/auto-prompt-sam2/stage6/notebook_logs/1793451239320667474_prepare.log
official epoch 1 train 20 / 252 steps 3
official epoch 1 train 40 / 252 steps 6
official epoch 1 train 60 / 252 steps 8
official epoch 1 train 80 / 252 steps 10
official epoch 1 train 100 / 252 steps 14
official epoch 1 train 120 / 252 steps 17
official epoch 1 train 140 / 252 steps 20
official epoch 1 train 160 / 252 steps 24
official epoch 1 train 180 / 252 steps 28
official epoch 1 train 200 / 252 steps 31
official epoch 1 train 220 / 252 steps 35
official epoch 1 train 240 / 252 steps 38
official 1 val water IoU 0.3532127303623637 contributors 161
Evidence: /content/drive/MyDrive/auto-prompt-sam2/regimes/official/stage6/0f8f08fb38e4e459203d37c4d86138a9273ac31f8090d2d1d97278cf98288904/evidence.zip SHA256: 40dcb7ca88bfe81f5aeac4306adc66c0c142e91f64386a154ccce4cda22a10bb
leave_paraguay_out epoch 1 train 20 / 213 steps 2
leave_paraguay_out epoch 1 train 40 / 213 steps 5
leave_p

## 3. Nuevo runtime: reanudar
En otro runtime GPU, ejecutar SOLO la celda de entorno y esta celda. Conservar salidas de prepare. Misma raíz, SHA y código; anterior escritor detenido. Verifica estados y repetición del siguiente paso, revierte diagnóstico y completa presupuesto. Ante interrupción posterior repetir resume; se repite época no comprometida.


In [2]:
command([str(PYTHON), str(REPO / 'scripts/run_stage6.py'), 'resume', '--root', PERSIST_ROOT], 'resume.log')


Log persistente: /content/drive/MyDrive/auto-prompt-sam2/stage6/notebook_logs/1793564984828106079_resume.log
official epoch 2 train 20 / 252 steps 44
official epoch 2 train 40 / 252 steps 47
official epoch 2 train 60 / 252 steps 50
official epoch 2 train 80 / 252 steps 53
official epoch 2 train 100 / 252 steps 56
official epoch 2 train 120 / 252 steps 59
official epoch 2 train 140 / 252 steps 62
official epoch 2 train 160 / 252 steps 66
official epoch 2 train 180 / 252 steps 69
official epoch 2 train 200 / 252 steps 72
official epoch 2 train 220 / 252 steps 75
official epoch 2 train 240 / 252 steps 79
official 2 val water IoU 0.40013167676627237 contributors 161
official epoch 3 train 20 / 252 steps 86
official epoch 3 train 40 / 252 steps 88
official epoch 3 train 60 / 252 steps 91
official epoch 3 train 80 / 252 steps 95
official epoch 3 train 100 / 252 steps 97
official epoch 3 train 120 / 252 steps 101
official epoch 3 train 140 / 252 steps 104
official epoch 3 train 160 / 252 step

## 4. Exportar evidencia y detener
Compartir notebook con salidas, ambos evidence.zip y SHA, logs/failure si existen. Checkpoints quedan en Drive con inventario; facilitar referencias para revisión. No cerrar etapa sin revisión externa.


In [3]:
for regime in ('official', 'leave_paraguay_out'):
    for folder in sorted((Path(PERSIST_ROOT) / 'regimes' / regime / 'stage6').iterdir()):
        report = folder / 'evidence/validation.json'
        if report.is_file():
            value = json.loads(report.read_text())
            print(regime, value['status'], 'best:', value['progress']['best_epoch'], value['progress']['best_iou'])
            print('Recovery:', value['progress']['recovery'])
            archive = folder / 'evidence.zip'
            print(archive, hashlib.sha256(archive.read_bytes()).hexdigest())


official computation_complete_pending_external_review best: 10 0.4840668324093302
Recovery: {'current_boot_id': 'd64509fb-d52a-4372-a067-97df906f2b25', 'different_runtime': True, 'next_epoch': 2, 'next_step_replay_bitwise_equal': True, 'previous_boot_id': '323cb76a-f894-4b8d-bb74-d43771680860', 'state_bitwise_equal': True}
/content/drive/MyDrive/auto-prompt-sam2/regimes/official/stage6/0f8f08fb38e4e459203d37c4d86138a9273ac31f8090d2d1d97278cf98288904/evidence.zip 9c8a1ac685ad0a9374d5a72d1214f7f8fa8b741f7f8580d2c6cc512113ffdd68
leave_paraguay_out computation_complete_pending_external_review best: 8 0.5058154025637092
Recovery: {'current_boot_id': 'd64509fb-d52a-4372-a067-97df906f2b25', 'different_runtime': True, 'next_epoch': 2, 'next_step_replay_bitwise_equal': True, 'previous_boot_id': '323cb76a-f894-4b8d-bb74-d43771680860', 'state_bitwise_equal': True}
/content/drive/MyDrive/auto-prompt-sam2/regimes/leave_paraguay_out/stage6/a14f5849a39a426944a8b301086f7b262f3a91b82fc7f6fd02fae3c3e080